# Description

This Master notebook is a pipeline to extracting and validating profiles with emails using APIS of PhantomBustor, DropContact, Googlesheets and HunterIO.

In [1]:
#!sudo /bin/bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

In [15]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [18]:
import logging
import os

import ck_marketing.hunterio.hunter_api as cmhuhuap
import ck_marketing.linkedin as cmliprfi
import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint
#from ck_marketing.process_automation.workflows import GoogleSheetsHelper, HunterIO
from ck_marketing.linkedin import Phantom

In [20]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

DEBUG:helpers.hsystem:> (git branch --show-current) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --short HEAD) 2>&1
DEBUG:helpers.hsystem:> (git log --date=local --oneline --graph --date-order --decorate --pretty=format:'%h %<(8)%aN%  %<(65)%s (%>(14)%ar) %ad %<(10)%d' -3) 2>&1


INFO:__main__:# Git
  branch_name='CmampTask11020_Compute_yamm_stats'
  hash='f1e22dc12'
  # Last commits:
    * f1e22dc12 GP Saggese Update                                                            (68 minutes ago) Tue Dec 31 17:55:47 2024  (HEAD -> CmampTask11020_Compute_yamm_stats, origin/CmampTask11020_Compute_yamm_stats)
    * 9e78ec83f GP Saggese Update                                                            (   2 hours ago) Tue Dec 31 16:53:51 2024           
    * 27ca5e75b GP Saggese Update                                                            (   3 hours ago) Tue Dec 31 15:53:29 2024           
# Machine info
  system=Linux
  node name=ce6cfb94b060
  release=6.6.22-linuxkit
  version=#1 SMP Fri Mar 29 12:21:27 UTC 2024
  machine=aarch64
  processor=aarch64
  cpu count=8
  cpu freq=None
  memory=svmem(total=8222072832, available=6157987840, percent=25.1, used=1848492032, free=2804412416, active=2223534080, inactive=2519330816, buffers=333570048, cached=3235598336, sha

# PhantomBuster - Extract Profiles

In [23]:
# Get the API keys from the environment variables.
phantom_api_key = os.getenv("Phantom_API_KEY")
hunter_api_key = os.getenv("Hunter_API_KEY")
dropcontact_api_key = os.getenv("Drop_API_KEY")

In [26]:
# Initialize the Phantom instance.
phantom = Phantom()

In [27]:
os.environ["linkedin_cookie"] = "AQEDAQB9OtQAcDLoAAABlB1XqBEAAAGUQWQsEU4AHARvrhO4I50n_SrQGhyC--tENUDpB5p0Nrkj8SX2RJgGsZ4VBNVpVh5CA4JV1a3ThO8hrKIR1hVpWcSMQLH-rz3qf7OSpsoOiBywfFx3IV3pca2g"

In [28]:
sales_nav = os.getenv("sales_nav_query")
link = os.getenv("linkedin_cookie")
agent_name = 'Sequoia Capital'
phantom.create_sales_nav_phantom(agent_name, sales_nav, link)

{'id': '7840163913587134'}

In [29]:
# Get and print all agents and their IDs.
agents = phantom.get_all_agents()
print("List of all agents and their IDs:\n")
for agent in agents:
    print(f"Agent Name: {agent['name']}, Agent ID: {agent['id']}")

List of all agents and their IDs:

Agent Name: Sequoia Capital, Agent ID: 7840163913587134
Agent Name: Untitled LinkedIn Auto Connect, Agent ID: 842771612674527
Agent Name: Untitled LinkedIn Profile Scraper, Agent ID: 2849719991221032
Agent Name: Untitled LinkedIn Connections Export, Agent ID: 2228606163430093


In [55]:
# Get agent ID and name.
AGENT_ID = "4773837904519897"

In [56]:
specific_agent_name = phantom.get_agent_name(AGENT_ID)
print(f"Selected Phantom: {specific_agent_name}")

Selected Phantom: Sequoia Capital


In [ ]:
# Launch the agent and get the results in a DataFrame.
df = phantom.launch_and_get_df(AGENT_ID)
print("DataFrame is fetched")

In [ ]:
# Google Drive Setup.
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)
drive_folder_id = "1fpsUZ8Nd52FGKxOuqVf11hzEetwvVoDq"
sheet_name = f"{specific_agent_name}_search_export"
tab_name = "search_export"

In [ ]:
file_id = google_sheet_helper.create_new_sheet_from_df(
    df, sheet_name, drive_folder_id, tab_name
)

# Clean Profiles

In [ ]:
words = ["talent"]
filtered_df = cmliprfi.filter_df(df, "title", words, "remove")

In [ ]:
sheet = google_sheet_helper.google_account.open_by_key(file_id)
title_clean = "cleaned_profiles"
cleaned_profiles_tab = sheet.add_worksheet(
    title=title_clean, rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, filtered_df, title_clean)

# HunterIO & Dropcontact - Extract emails

In [ ]:
merged_df = cmhuhuap.hunter_drop_emails(
    "firstName",
    "lastName",
    "companyName",
    "Sheet1",
    hunter_api_key,
    google_creds_path,
    file_id,
    dropcontact_api_key,
)

# HunterIO - Verify emails

In [ ]:
hunter_instance = HunterIO(hunter_api_key)
verified_df = hunter_instance.verify_emails(merged_df, "all_emails")

In [ ]:
cleaned_profiles_tab = sheet.add_worksheet(
    title="hunter_verification", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, verified_df, "hunter_verification")

# Final dataframe

In [ ]:
final_df = verified_df[
    [
        "fullName",
        "firstName",
        "lastName",
        "profileUrl",
        "title",
        "all_emails",
        "hunter_verification",
    ]
]
# Step 2: Filter out rows where 'hunter_extracted_email' is empty.
final_df = final_df[
    final_df["all_emails"].notna() & (final_df["all_emails"] != "")
]

cleaned_profiles_tab = sheet.add_worksheet(
    title="final_df", rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, final_df, "final_df")

#  Delete Phantom

In [57]:
phantom.delete_phantom(AGENT_ID)

INFO:ck_marketing.linkedin.phantom_api.phantombustorrrr:Successfully deleted phantom 4773837904519897. Response: OK
